# 00 — Retile Polygons (100 m² hex)

Cut the 65 source polygons into 100 m² hexagonal tiles — one Sentinel-2 pixel per tile.  
Output: `../data/polygons/tiles_100m2.pkl` and `tiles_100m2.geojson`

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon

PLOTS_PATH = '../data/polygons/RegressionRidge_smol.geojson'
OUT_PKL    = '../data/polygons/tiles_100m2.pkl'
OUT_GJ     = '../data/polygons/tiles_100m2.geojson'

MIN_AREA = 100  # m²  (vs 1000 m² in v1) — native Sentinel-2 10m pixel


/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
plots = gpd.read_file(PLOTS_PATH).to_crs(26910)  # UTM for meter-based work
print(f'Source polygons: {len(plots)}, total area: {plots.geometry.area.sum():.0f} m²')


Source polygons: 65, total area: 3160720 m²


In [3]:
def make_hex(cx, cy, r):
    angles = np.deg2rad([0, 60, 120, 180, 240, 300])
    return Polygon([(cx + r*np.cos(a), cy + r*np.sin(a)) for a in angles])

def clean_geom(g, min_area):
    if g.is_empty:
        return None
    g = g.buffer(0)
    if g.is_empty:
        return None
    if isinstance(g, MultiPolygon):
        g = max(g.geoms, key=lambda x: x.area)
    return g if g.area >= min_area else None

def subdivide_hex(poly, min_area_m2):
    r     = np.sqrt((2 * min_area_m2) / (3 * np.sqrt(3)))
    horiz = 1.5 * r
    vert  = np.sqrt(3) * r
    minx, miny, maxx, maxy = poly.bounds
    hexes, row = [], 0
    y = miny
    while y <= maxy + vert:
        x = minx + (0.75 * r if row % 2 else 0)
        while x <= maxx + horiz:
            clipped = poly.intersection(make_hex(x, y, r))
            g = clean_geom(clipped, min_area_m2 * 0.25)
            if g is not None:
                hexes.append(g)
            x += horiz
        y += vert
        row += 1
    return hexes


In [4]:
rows = []
for idx, row in plots.iterrows():
    parent = row.get('Name', f'plot_{idx}')
    for geom in subdivide_hex(row.geometry, MIN_AREA):
        rows.append({'parent_plot': parent, 'geometry': geom})

tiles = gpd.GeoDataFrame(rows, crs=plots.crs)
tiles.insert(0, 'tile_id', range(len(tiles)))
tiles = tiles.to_crs(4326)

print(f'Tiles created: {len(tiles)}')
areas = tiles.to_crs(26910).geometry.area
print(f'Tile area — median: {areas.median():.1f} m²  min: {areas.min():.1f}  max: {areas.max():.1f}')


Tiles created: 32978
Tile area — median: 100.0 m²  min: 25.0  max: 100.0


In [5]:
tiles.to_pickle(OUT_PKL)
tiles.to_file(OUT_GJ, driver='GeoJSON')
print(f'Saved → {OUT_PKL}')
print(f'Saved → {OUT_GJ}')
tiles.head()


Saved → ../data/polygons/tiles_100m2.pkl
Saved → ../data/polygons/tiles_100m2.geojson


,tile_id,parent_plot,geometry
0,0,plot_0,"POLYGON ((-119.78208 45.87245, -119.78214 45.8..."
1,1,plot_0,"POLYGON ((-119.78204 45.87245, -119.78200 45.8..."
2,2,plot_0,"POLYGON ((-119.78191 45.87245, -119.78188 45.8..."
3,3,plot_0,"POLYGON ((-119.78179 45.87245, -119.78176 45.8..."
4,4,plot_0,"POLYGON ((-119.78242 45.87259, -119.78241 45.8..."
